# **Data Visualization with Pandas and Matplotlib**

While MatPlotLib is powerful alone, it is very commonly used with the Pandas library.

In this tutorial, we will use a set of CSV files to demonstrate how to use Panda alongside Matplotlib to create simple data visualizations.

## **Outline**
- Setup
- Basic Plots
- Adding Title and Labels
- Other Plots
- Customization
- Examples
	- Rearrange Bars
	- Time Series
	- Multiple-Traces

## **Setup**

In [ ]:
# Import Necessary Libraries

import numpy as np # For Calculations Purpose
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

In [ ]:
# Using the crime statistics data
crime = pd.read_csv('crime-new.csv',parse_dates=['occurrencedate'])
crime.head()

In [ ]:
crime.info()

## **Basic Plots**

Similar to Matplotlib, a simple plot can be created with the <code>plot()</code> method.

However, most plots requires only two variables, so the simplest plot can be created by chaining <code>.value_counts()</code> on one column then plotting it.

In [ ]:
# Finding the number of crimes for each Major Crime Indicator (MCI)
crime['MCI'].value_counts().plot();

## **Adding Title and Labels**

Just like in MatplolLib, the graphs can be more informative by adding titles and labels to the graph.

To add titles, a parameter of <code>title</code> in the <code>plot()</code> method.

In [ ]:
crime['MCI'].value_counts().plot(title = "Crime Indicator");

To add labels, it is the same as Matplotlib.

In [ ]:
crime['MCI'].value_counts().plot(title = "Crime Indicator")
plt.xlabel('MCI')
plt.ylabel('Frequency Count');

The orientation of the labels and the ticks can be rotated, this can be useful for longer names.

In [ ]:
crime['MCI'].value_counts().plot(title = "Crime Indicator")
plt.xlabel('MCI')
plt.xticks(rotation=90)
plt.ylabel('Frequency Count');

## **Other Plots**
Although the default is line graph, other kinds of graphs can be created using the <code>kind</code> parameter.

In [ ]:
# Bar Plot
crime['MCI'].value_counts().plot(kind = 'bar', title = "Crime Indicator")
plt.xlabel('MCI')
plt.ylabel('Frequency Count');

In [ ]:
# Horizontal Bar Plot
crime['MCI'].value_counts().plot(kind = 'barh', title = "Crime Indicator")
plt.xlabel('MCI')
plt.ylabel('Frequency Count');

In [ ]:
# Pie Chart
crime['MCI'].value_counts().plot(kind = 'pie', title = "Crime Indicator")
plt.xlabel('MCI')
plt.ylabel('Frequency Count');

## **Customization**

There are a few things we can do to customize the graphs.

In [ ]:
# Sorting the values

crime['MCI'].value_counts().sort_values().plot(kind = 'barh', title = "Crime Indicator")
plt.xlabel('MCI')
plt.ylabel('Frequency Count');

The colors of the bar plots can be customized by creating a list of colors.

If the number of colors is less than the number of bars, then the pattern repeats.

In [ ]:
# Customizing the colors
colours = ['lightgrey', 'lightgrey', 'lightgrey', 'lightgrey', 'blue']
crime['MCI'].value_counts().sort_values().plot(kind = 'barh', title = "Crime Indicator",
                                              color=colours)
plt.xlabel('MCI')
plt.ylabel('Frequency Count');

## **Examples**

Below are a few examples of how to use Pandas and Matplotlib to create data visualizations.

### **Example 1: Rearrange Bars**

In the previous examples, you may have noticed that the bars are usually arranged by the order of appearance. And although we can change the order by using <code>sort_values()</code>, sometimes that is not the best way.

In [ ]:
# Grouping the data by day of week
crime['occurrencedayofweek'].value_counts()

In [ ]:
# Bar plot for this data
crime['occurrencedayofweek'].value_counts().plot(kind='bar');

Although the plot shows the data from most to least number of occurrences, it might be better to arrange the graph by the order of the day of the week.

To do that, we need to create a list for the order of the days we want.

In [ ]:
days = ['Sunday', 'Monday', 'Tuesday','Wednesday','Thursday','Friday','Saturday']

In [ ]:
# Making the days into the index
crime['occurrencedayofweek'].value_counts().reindex(days)

But why are all the counts NaN?

In [ ]:
# Investigate the occurrencedayofweek column values
crime['occurrencedayofweek'].values

Notice the values in the column have all these whitespace characters. In Python, whitespace characters are also considered when looking at the values. This means <code>'Monday'</code> and <code>'Monday   '</code> are treated as different objects.

In [ ]:
# Strip away the whitespace in the column
crime['occurrencedayofweek'].str.strip().values

In [ ]:
# Overwriting the original column
crime['occurrencedayofweek'] = crime['occurrencedayofweek'].str.strip()
crime['occurrencedayofweek'].values

In [ ]:
# Repeat the previous step to create the graph
crime_sorted = crime['occurrencedayofweek'].value_counts().reindex(days)
crime_sorted

In [ ]:
crime_sorted.plot(kind='bar');

### **Example 2: Time Series**
A time series plot is an application of line plot. The only condition is that the x-values (independent variable) is a datetime object.

To plot time series, the datetime column would be set as the index.

In [ ]:
# In the dataframe, the occurrencedate column can be used as the datetime column
crime['occurrencedate'].head()

In [ ]:
# Simply plotting the data with occurrencedayofyear
crime.set_index('occurrencedate')['occurrencedayofyear'].plot(kind='line');

Looking at this graph, it is very messy and needs some cleaning.

Instead of plotting by the individual dates, it better to group the data by the month.

In [ ]:
# Grouping the MCI occurrence by year and month

crime_date = crime.groupby(['occurrenceyear','occurrencemonth'], as_index=False)\
                    .agg({'occurrencedate':'min', 'MCI':'count'}) \
                    .rename(columns = {'MCI':'count'})\
                    .sort_values('occurrencedate')

crime_date

In [ ]:
crime_date.set_index('occurrencedate')['count'].plot();

The graph is much more organized, but there seems to be some unnecessary data before 2014.

Let's remove those to get a better graph.

In [ ]:
# Filter df so we have >= 2014
crime_date_recent = crime_date[crime_date['occurrenceyear'] >= 2014]
crime_date_recent.set_index('occurrencedate')['count'].plot();

### **Example 3: Multiple-Traces**

Just as with Matplotlib, multiple graphs can be layered on top of each other.

In [ ]:
# Filter out the unnecessary data (prior to 2014)
crime2 = crime[crime['occurrenceyear']>=2014]
crime2

In [ ]:
# Creating a robbery data
robbery_date = (crime2[crime2['MCI'] == 'Robbery'] # Filter only the Robbery data
               .groupby(['occurrenceyear','occurrencemonth'], as_index=False) # Grouping by the year and month
               .agg({'occurrencedate': 'min', 'MCI':'count'}) # Aggregate the MCI counts
               .rename(columns={'MCI':'count'}) # Renaming the column
               .sort_values('occurrencedate')) # Organizing the data

robbery_date

In [ ]:
# Repeat for auto theft data
auto_date = (crime2[crime2['MCI'] == 'Auto Theft']
               .groupby(['occurrenceyear','occurrencemonth'], as_index=False)
               .agg({'occurrencedate': 'min', 'MCI':'count'})
               .rename(columns={'MCI':'count'})
               .sort_values('occurrencedate'))

auto_date

In [ ]:
robbery_date.set_index('occurrencedate')['count'].plot(label='Robbery')
auto_date.set_index('occurrencedate')['count'].plot(label='Auto Theft');